# ViFeedback — Kaggle training (robust version v2)

This notebook is a thin Kaggle runner for the `vifeedback` repository. It is designed to survive
the common Kaggle failure modes that previously caused late crashes:

- repository supplied as either an extracted Kaggle Dataset or a ZIP;
- stale `vifeedback` modules left in the live Python kernel after re-running install cells;
- package versions that do **not** provide `vifeedback.check_import()`;
- repository versions that do **not** provide `vifeedback.evaluation.report`;
- Kaggle's preinstalled `google-adk` missing `google-cloud-bigquery-storage`;
- shell commands that fail without stopping a notebook cell.

Results still use the repository's own `results/registry.csv` when available.

## Before you run anything

| Step | Where |
|---|---|
| 1. Accelerator → **GPU** | *Settings* panel, right side |
| 2. Attach the `vifeedback-repo` Dataset | *Add Input* |
| 3. Enable Internet if model/data downloads are needed | *Settings* |
| 4. Run from the top after changing the attached Dataset version | Notebook |


---

## 1. Verify the environment


In [7]:
import shutil
import socket
import subprocess
import torch

# GPU diagnostics: do not fail just because nvidia-smi formatting changes.
if shutil.which('nvidia-smi'):
    subprocess.run([
        'nvidia-smi',
        '--query-gpu=name,memory.total,driver_version',
        '--format=csv,noheader',
    ], check=False)

assert torch.cuda.is_available(), (
    'No CUDA GPU is available. In Kaggle: Settings -> Accelerator -> select a GPU, '
    'then restart/re-run the notebook.'
)

p = torch.cuda.get_device_properties(0)
vram_gb = p.total_memory / (1024**3)
print(f'{p.name}  {vram_gb:.2f} GiB  capability {p.major}.{p.minor}  torch {torch.__version__}')

# The planned models need substantially more than a 4 GB laptop GPU. Keep this as a warning
# instead of a hard assertion so the notebook can still be used for smaller/debug runs.
if vram_gb < 10:
    print(f'WARNING: only {vram_gb:.1f} GiB VRAM; large-model cells may OOM. '
          'Reduce batch size / use grad accumulation or choose a larger Kaggle GPU.')

try:
    socket.create_connection(('huggingface.co', 443), timeout=5).close()
    print('internet: OK')
except OSError:
    print('WARNING: huggingface.co is not reachable. Hub/data download steps may fail. '
          'Enable Internet in Kaggle Settings unless all required assets are already cached.')


Tesla T4, 15360 MiB, 580.159.04
Tesla T4, 15360 MiB, 580.159.04
Tesla T4  14.56 GiB  capability 7.5  torch 2.10.0+cu128
internet: OK


### Optional: Hugging Face token

Unauthenticated Hub downloads are rate-limited. If you added an `HF_TOKEN` secret
(*Add-ons → Secrets*), this picks it up; if not, it carries on unauthenticated.


In [8]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print(f'No HF_TOKEN ({type(e).__name__}) — continuing unauthenticated, which is fine')

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['PYTHONUNBUFFERED'] = '1'


No HF_TOKEN (BackendError) — continuing unauthenticated, which is fine


---

## 2. Prepare the repository

This cell accepts either:

1. an **extracted** repository under `/kaggle/input/...`, or
2. a repository **ZIP** anywhere under `/kaggle/input`.

It deliberately copies/extracts to `/kaggle/working/repo` (not a directory named
`vifeedback`) so that the work directory itself can never shadow the Python package.


In [9]:
import glob
import os
import pathlib
import shutil
import subprocess
import zipfile

REPO_URL = ''  # Optional, e.g. 'https://github.com/<user>/ViFeedback-NLP-Service.git'

WORK = pathlib.Path('/kaggle/working/repo')
assert WORK.name != 'vifeedback', 'WORK must not be named vifeedback because it can shadow the package'

def is_repo_root(p: pathlib.Path) -> bool:
    return (
        (p / 'pyproject.toml').is_file()
        and (p / 'src' / 'vifeedback').is_dir()
    )

def find_repo_roots(base: pathlib.Path):
    roots = []
    if is_repo_root(base):
        roots.append(base)
    for pyproject in base.rglob('pyproject.toml'):
        root = pyproject.parent
        if is_repo_root(root):
            roots.append(root)
    # Remove duplicates while preserving a deterministic shallow-first order.
    unique = {p.resolve(): p for p in roots}
    return sorted(unique.values(), key=lambda p: (len(p.resolve().parts), str(p)))

if WORK.exists():
    shutil.rmtree(WORK)

if REPO_URL:
    print('cloning:', REPO_URL)
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(WORK)], check=True)
else:
    input_root = pathlib.Path('/kaggle/input')

    # Prefer an already-extracted repo. This avoids accidentally selecting an unrelated ZIP
    # from another attached Kaggle dataset.
    extracted = find_repo_roots(input_root)
    if extracted:
        src_repo = extracted[0]
        print('using extracted repo:', src_repo)
        shutil.copytree(src_repo, WORK)
    else:
        zips = sorted(input_root.rglob('*.zip'))
        assert zips, (
            'No ViFeedback repository was found under /kaggle/input. '
            'Attach the vifeedback-repo Dataset (extracted or ZIP) and run again.'
        )

        # Try ZIPs one-by-one until one actually contains pyproject.toml + src/vifeedback.
        chosen = None
        staging = pathlib.Path('/kaggle/working/_vifeedback_unpack')
        for zpath in zips:
            if staging.exists():
                shutil.rmtree(staging)
            staging.mkdir(parents=True)
            try:
                with zipfile.ZipFile(zpath) as zf:
                    zf.extractall(staging)
            except zipfile.BadZipFile:
                continue

            roots = find_repo_roots(staging)
            if roots:
                chosen = (zpath, roots[0])
                break

        assert chosen is not None, (
            'ZIP files were found under /kaggle/input, but none contained '
            'pyproject.toml together with src/vifeedback.'
        )
        zpath, src_repo = chosen
        print('using zip:', zpath)
        print('repo root inside zip:', src_repo)
        shutil.copytree(src_repo, WORK)
        shutil.rmtree(staging, ignore_errors=True)

os.chdir(WORK)

assert (WORK / 'pyproject.toml').is_file(), f'pyproject.toml missing in {WORK}'
assert (WORK / 'src' / 'vifeedback').is_dir(), (
    f'{WORK / "src" / "vifeedback"} is missing; the attached Dataset is not a usable source checkout.'
)
assert (WORK / 'src' / 'vifeedback' / 'cli.py').is_file() or (WORK / 'src' / 'vifeedback' / 'cli').exists(), (
    'vifeedback CLI source is missing from the attached repository.'
)

evaluation_report = WORK / 'src' / 'vifeedback' / 'evaluation' / 'report.py'
if not evaluation_report.exists():
    print(
        'NOTE: src/vifeedback/evaluation/report.py is not present in this repository version. '
        'That is OK: the verification cell will read results/registry.csv directly.'
    )

print('cwd:', pathlib.Path.cwd())
print('top-level:', sorted(x.name for x in WORK.iterdir()))


using extracted repo: /kaggle/input/datasets/datthnh/vifeedback-repo
cwd: /kaggle/working/repo
top-level: ['.gitignore', 'README.md', 'data', 'docs', 'notebooks', 'pyproject.toml', 'results', 'src', 'tests']


### Install

The project itself is installed with `--no-deps` so pip cannot replace Kaggle's CUDA-enabled
PyTorch. Required runtime packages are installed explicitly.

Kaggle images that include `google-adk==1.29.0` can be missing
`google-cloud-bigquery-storage`; this notebook installs that dependency explicitly so pip no
longer reports the resolver conflict shown in the previous run.


In [10]:
import importlib
import importlib.util
import os
import pathlib
import subprocess
import sys

def pip_install(*packages, upgrade=False, no_deps=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check']
    if upgrade:
        cmd.append('-U')
    if no_deps:
        cmd.append('--no-deps')
    cmd.extend(packages)
    print('>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

# 1) Install the local project without allowing pip to replace Kaggle's CUDA torch.
pip_install('-e', '.', no_deps=True)

# 2) Fix the known Kaggle google-adk dependency gap BEFORE installing the remaining packages.
pip_install('google-cloud-bigquery-storage>=2.0.0')

# 3) Explicit runtime/test dependencies used by this notebook.
pip_install(
    'typer',
    'pyyaml',
    'py-cpuinfo',
    'pyvi',
    'underthesea',
    'pytest',
)

# Keep the Hugging Face stack modern enough for the repository, but do not upgrade torch.
pip_install(
    'transformers>=4.44',
    'datasets>=3.0',
    'huggingface-hub',
    upgrade=True,
)

SRC_PATH = (pathlib.Path.cwd() / 'src').resolve()
SRC = str(SRC_PATH)

# A live Kaggle kernel can retain an OLD vifeedback module in sys.modules even after sys.path
# changes. In that case Python keeps using the stale package and may claim that new submodules
# do not exist. Remove every cached module from this package before importing it again.
stale = [name for name in list(sys.modules)
         if name == 'vifeedback' or name.startswith('vifeedback.')]
for name in stale:
    del sys.modules[name]
if stale:
    print(f'cleared {len(stale)} cached vifeedback module(s) from sys.modules')

if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
importlib.invalidate_caches()

spec = importlib.util.find_spec('vifeedback')
assert spec is not None, f'Python cannot find vifeedback after adding {SRC} to sys.path'

import vifeedback

module_file = getattr(vifeedback, '__file__', None)
assert module_file, (
    'vifeedback resolved as a namespace package instead of the repository package. '
    f'spec={spec!r}'
)
module_path = pathlib.Path(module_file).resolve()
expected_pkg = (SRC_PATH / 'vifeedback').resolve()
assert expected_pkg == module_path.parent or expected_pkg in module_path.parents, (
    'Imported the wrong vifeedback package.\n'
    f'Expected under: {expected_pkg}\n'
    f'Actually loaded: {module_path}'
)

# Do NOT call vifeedback.check_import(): that helper is not part of every repository version.
print('vifeedback loaded from:', module_path)

# The notebook trains through the CLI. Verify that API now, before any long GPU run.
cli_spec = importlib.util.find_spec('vifeedback.cli')
assert cli_spec is not None, 'vifeedback.cli is missing from this repository version'
print('vifeedback.cli:', cli_spec.origin)

# evaluation.report is optional here. Some repository versions do not have it; the results
# verification cell below has a direct registry.csv fallback.
try:
    report_spec = importlib.util.find_spec('vifeedback.evaluation.report')
except (ModuleNotFoundError, AttributeError):
    report_spec = None

if report_spec is None:
    print('vifeedback.evaluation.report: not available -> registry.csv fallback will be used')
else:
    print('vifeedback.evaluation.report:', report_spec.origin)

# Every later subprocess must resolve the same source tree first.
old_pythonpath = os.environ.get('PYTHONPATH', '')
parts = [p for p in old_pythonpath.split(os.pathsep) if p and p != SRC]
os.environ['PYTHONPATH'] = os.pathsep.join([SRC] + parts)

# Verify the exact interpreter/module path that later training cells will use.
subprocess.run([sys.executable, '-m', 'vifeedback.cli', '--help'], check=True)

# Diagnostic only: Kaggle base images can contain unrelated package conflicts. The known
# google-adk / bigquery-storage conflict should no longer appear after the install above.
check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
if check.returncode == 0:
    print('pip check: OK')
else:
    print('pip check reported remaining environment conflicts:')
    print(check.stdout.strip())
    print('These are not hidden; inspect them before a long training run if they mention vifeedback, torch, transformers, or datasets.')


> /usr/bin/python3 -m pip install -q --disable-pip-version-check --no-deps -e .
> /usr/bin/python3 -m pip install -q --disable-pip-version-check google-cloud-bigquery-storage>=2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.0/308.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 74.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=

> /usr/bin/python3 -m pip install -q --disable-pip-version-check typer pyyaml py-cpuinfo pyvi underthesea pytest
> /usr/bin/python3 -m pip install -q --disable-pip-version-check -U transformers>=4.44 datasets>=3.0 huggingface-hub
cleared 1 cached vifeedback module(s) from sys.modules
vifeedback loaded from: /kaggle/working/repo/src/vifeedback/__init__.py
vifeedback.cli: /kaggle/working/repo/src/vifeedback/cli.py
vifeedback.evaluation.report: /kaggle/working/repo/src/vifeedback/evaluation/report.py
                                                                                
 Usage: python -m vifeedback.cli [OPTIONS] COMMAND [ARGS]...                    
                                                                                
 ViFeedback — Vietnamese feedback classification                                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help   

---

## 3. Data

Fetches UIT-VSFC and asserts the official split sizes (11,426 / 1,583 / 3,166) plus the leakage
figures. If the upstream ever shifts, this fails here rather than producing numbers against
different data.


In [11]:
import subprocess
import sys

# Run through subprocess so failures stop the cell immediately (plain !commands can make it easy
# to miss an earlier non-zero exit code in a long output stream).
subprocess.run([sys.executable, '-m', 'vifeedback.cli', 'data', 'fetch'], check=True)

TESTS_DATA = WORK / 'tests' / 'data'
if TESTS_DATA.exists():
    subprocess.run([sys.executable, '-m', 'pytest', str(TESTS_DATA), '-q'], check=True)
else:
    print(f'WARNING: {TESTS_DATA} does not exist; skipping repository data tests.')


  train       11,426 rows  sha256=c0fdba92766e2cb3...
  validation   1,583 rows  sha256=2c85203815cb2d5c...
  test         3,166 rows  sha256=a2b7beaa5a6ff745...
  source: uitnlp/vietnamese_students_feedback@refs/convert/parquet
..............                                                           [100%]


Build the segmentation variant. Segmentation is worth **+0.023 macro-F1** (ADR-012), so training
on raw text here would not be comparable to the laptop results. `pyvi` is the serving choice and
is pure Python, so no JVM is needed.


In [12]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'vifeedback.cli', 'data', 'segmenters'], check=True)
subprocess.run([
    sys.executable, '-m', 'vifeedback.cli', 'data', 'variants', '--name', 'seg_pyvi'
], check=True)


  OK  none           identity
  OK  underthesea    importable
  OK  pyvi           importable
  --  vncorenlp      py_vncorenlp not installed
variant                 cond  changed%  underscores  tok.reduction  ms/sentence
-------------------------------------------------------------------------------
seg_pyvi                 P2b    92.0%       35,178         21.5%        0.190


CompletedProcess(args=['/usr/bin/python3', '-m', 'vifeedback.cli', 'data', 'variants', '--name', 'seg_pyvi'], returncode=0)

---

## 4. Train

**Runtime estimates** (P100, 5 seeds, 4 epochs). Kaggle sessions run up to 9 h and the free quota
is 30 GPU-hours/week, so run the cells you need rather than all of them.

| Cell | Model | Est. total |
|---|---|---|
| 4a | `phobert-large` | **~75 min** |
| 4b | `xlm-roberta-base` full | ~40 min |
| 4c | `CafeBERT` | ~90 min |

### On `--grad-accum`

`phobert-large` needs batch 16 to fit comfortably, but `phobert-base` was trained at 32. Comparing
them at different effective batch sizes would not be a controlled comparison, so `--grad-accum 2`
restores the effective batch to 32. The CLI prints the effective batch it is using — check it.


In [13]:
# 4a — PhoBERT-large.
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'vifeedback.cli', 'train', 'run',
    '--task', 'sentiment',
    '--model', 'phobert-large',
    '--recipe', 'base',
    '--preprocessing', 'seg_pyvi',
    '--seeds', 'all',
    '--epochs', '4',
    '--lr', '1e-5',
    '--batch-size', '16',
    '--grad-accum', '2',
    '--max-length', '96',
    '--phase', '4',
], check=True)


  effective batch = 16 x 2 = 32

[p4-sent-phobert-large-seg_pyvi-base-s42-val]  phobert-large / sentiment / base / seed 42


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7728.78it/s]


  epoch 1/4  loss 0.4600  dev macro-F1 0.8604  wtd 0.9513  acc 0.9539  neutral F1 0.650  (132s)
  epoch 2/4  loss 0.1733  dev macro-F1 0.8411  wtd 0.9481  acc 0.9514  neutral F1 0.593  (267s)
  epoch 3/4  loss 0.1220  dev macro-F1 0.8488  wtd 0.9511  acc 0.9539  neutral F1 0.612  (401s)
  epoch 4/4  loss 0.0856  dev macro-F1 0.8583  wtd 0.9527  acc 0.9551  neutral F1 0.639  (534s)
  early stopping at epoch 4 (best was 1)
  --> validation
  Macro-F1      0.860   [95% CI 0.824-0.895]
  Weighted F1   0.951
  Accuracy      0.954
  Bal. accuracy 0.828
  MCC           0.914
  Per class     negative          P 0.949  R 0.976  F1 0.962  (n=  705, predicted   725)
                neutral           P 0.830  R 0.534  F1 0.650  (n=   73, predicted    47)
                positive          P 0.965  R 0.973  F1 0.969  (n=  805, predicted   811)

[p4-sent-phobert-large-seg_pyvi-base-s1337-val]  phobert-large / sentiment / base / seed 1337


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 12646.08it/s]


  epoch 1/4  loss 0.4584  dev macro-F1 0.8145  wtd 0.9407  acc 0.9450  neutral F1 0.522  (134s)
  epoch 2/4  loss 0.1669  dev macro-F1 0.8378  wtd 0.9458  acc 0.9482  neutral F1 0.587  (268s)
  epoch 3/4  loss 0.1222  dev macro-F1 0.8457  wtd 0.9495  acc 0.9526  neutral F1 0.605  (403s)
  epoch 4/4  loss 0.0952  dev macro-F1 0.8520  wtd 0.9507  acc 0.9533  neutral F1 0.623  (537s)
  --> validation
  Macro-F1      0.852   [95% CI 0.815-0.885]
  Weighted F1   0.951
  Accuracy      0.953
  Bal. accuracy 0.823
  MCC           0.913
  Per class     negative          P 0.951  R 0.982  F1 0.966  (n=  705, predicted   728)
                neutral           P 0.776  R 0.521  F1 0.623  (n=   73, predicted    49)
                positive          P 0.967  R 0.968  F1 0.967  (n=  805, predicted   806)

[p4-sent-phobert-large-seg_pyvi-base-s2024-val]  phobert-large / sentiment / base / seed 2024


...

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 8644.11it/s]


  epoch 1/4  loss 0.4629  dev macro-F1 0.8413  wtd 0.9460  acc 0.9482  neutral F1 0.598  (133s)
  epoch 2/4  loss 0.1662  dev macro-F1 0.8075  wtd 0.9421  acc 0.9482  neutral F1 0.495  (268s)
  epoch 3/4  loss 0.1188  dev macro-F1 0.8483  wtd 0.9488  acc 0.9520  neutral F1 0.615  (403s)
  epoch 4/4  loss 0.0875  dev macro-F1 0.8534  wtd 0.9506  acc 0.9533  neutral F1 0.628  (539s)
  --> validation
  Macro-F1      0.853   [95% CI 0.817-0.888]
  Weighted F1   0.951
  Accuracy      0.953
  Bal. accuracy 0.823
  MCC           0.913
  Per class     negative          P 0.953  R 0.976  F1 0.964  (n=  705, predicted   722)
                neutral           P 0.792  R 0.521  F1 0.628  (n=   73, predicted    48)
                positive          P 0.963  R 0.973  F1 0.968  (n=  805, predicted   813)

[p4-sent-phobert-large-seg_pyvi-base-s31337-val]  phobert-large / sentiment / base / seed 31337


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 8492.08it/s]


  epoch 1/4  loss 0.4686  dev macro-F1 0.7724  wtd 0.9284  acc 0.9349  neutral F1 0.411  (134s)
  epoch 2/4  loss 0.1784  dev macro-F1 0.8551  wtd 0.9502  acc 0.9526  neutral F1 0.634  (269s)
  epoch 3/4  loss 0.1249  dev macro-F1 0.8589  wtd 0.9510  acc 0.9533  neutral F1 0.645  (404s)
  epoch 4/4  loss 0.0963  dev macro-F1 0.8472  wtd 0.9495  acc 0.9526  neutral F1 0.610  (538s)
  --> validation
  Macro-F1      0.859   [95% CI 0.822-0.893]
  Weighted F1   0.951
  Accuracy      0.953
  Bal. accuracy 0.831
  MCC           0.913
  Per class     negative          P 0.955  R 0.973  F1 0.964  (n=  705, predicted   718)
                neutral           P 0.784  R 0.548  F1 0.645  (n=   73, predicted    51)
                positive          P 0.962  R 0.973  F1 0.967  (n=  805, predicted   814)

=== SENTIMENT / phobert-large / base / 5 seeds ===
  [validation]  n_seeds=5  best_epochs=[1, 4, 4, 4, 3]
    macro_f1       0.8560 ± 0.0036   [0.8520, 0.8604]
    weighted_f1    0.9509 ± 0.0003   [

CompletedProcess(args=['/usr/bin/python3', '-m', 'vifeedback.cli', 'train', 'run', '--task', 'sentiment', '--model', 'phobert-large', '--recipe', 'base', '--preprocessing', 'seg_pyvi', '--seeds', 'all', '--epochs', '4', '--lr', '1e-5', '--batch-size', '16', '--grad-accum', '2', '--max-length', '96', '--phase', '4'], returncode=0)

In [14]:
# 4b — XLM-R base with UNFROZEN embeddings.
# The frozen-embedding version fits the laptop; run that one there, not here.
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'vifeedback.cli', 'train', 'run',
    '--task', 'sentiment',
    '--model', 'xlmr-base',
    '--recipe', 'base',
    '--preprocessing', 'seg_pyvi',
    '--seeds', 'all',
    '--epochs', '4',
    '--batch-size', '32',
    '--max-length', '96',
    '--phase', '4',
], check=True)


  effective batch = 32 x 1 = 32

[p4-sent-xlmr-base-seg_pyvi-base-s42-val]  xlmr-base / sentiment / base / seed 42


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2512.05it/s]


  epoch 1/4  loss 0.4620  dev macro-F1 0.6284  wtd 0.8993  acc 0.9210  neutral F1 0.000  (79s)
  epoch 2/4  loss 0.2390  dev macro-F1 0.8181  wtd 0.9380  acc 0.9425  neutral F1 0.541  (158s)
  epoch 3/4  loss 0.1852  dev macro-F1 0.8393  wtd 0.9411  acc 0.9444  neutral F1 0.603  (238s)
  epoch 4/4  loss 0.1481  dev macro-F1 0.8363  wtd 0.9412  acc 0.9444  neutral F1 0.593  (317s)
  --> validation
  Macro-F1      0.839   [95% CI 0.800-0.875]
  Weighted F1   0.941
  Accuracy      0.944
  Bal. accuracy 0.804
  MCC           0.896
  Per class     negative          P 0.943  R 0.967  F1 0.955  (n=  705, predicted   723)
                neutral           P 0.814  R 0.479  F1 0.603  (n=   73, predicted    43)
                positive          P 0.952  R 0.966  F1 0.959  (n=  805, predicted   817)

[p4-sent-xlmr-base-seg_pyvi-base-s1337-val]  xlmr-base / sentiment / base / seed 1337


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2490.40it/s]


  epoch 1/4  loss 0.4708  dev macro-F1 0.7830  wtd 0.9261  acc 0.9330  neutral F1 0.451  (78s)
  epoch 2/4  loss 0.2488  dev macro-F1 0.8298  wtd 0.9336  acc 0.9375  neutral F1 0.589  (157s)
  epoch 3/4  loss 0.1834  dev macro-F1 0.8319  wtd 0.9386  acc 0.9425  neutral F1 0.584  (236s)
  epoch 4/4  loss 0.1468  dev macro-F1 0.8499  wtd 0.9453  acc 0.9476  neutral F1 0.629  (315s)
  --> validation
  Macro-F1      0.850   [95% CI 0.811-0.881]
  Weighted F1   0.945
  Accuracy      0.948
  Bal. accuracy 0.823
  MCC           0.902
  Per class     negative          P 0.947  R 0.969  F1 0.958  (n=  705, predicted   721)
                neutral           P 0.765  R 0.534  F1 0.629  (n=   73, predicted    51)
                positive          P 0.959  R 0.966  F1 0.963  (n=  805, predicted   811)

[p4-sent-xlmr-base-seg_pyvi-base-s2024-val]  xlmr-base / sentiment / base / seed 2024


...

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2285.95it/s]


  epoch 1/4  loss 0.4707  dev macro-F1 0.7488  wtd 0.9117  acc 0.9204  neutral F1 0.371  (78s)
  epoch 2/4  loss 0.2348  dev macro-F1 0.8188  wtd 0.9351  acc 0.9394  neutral F1 0.549  (158s)
  epoch 3/4  loss 0.1774  dev macro-F1 0.8320  wtd 0.9423  acc 0.9457  neutral F1 0.576  (237s)
  epoch 4/4  loss 0.1342  dev macro-F1 0.8329  wtd 0.9420  acc 0.9438  neutral F1 0.580  (316s)
  --> validation
  Macro-F1      0.833   [95% CI 0.794-0.866]
  Weighted F1   0.942
  Accuracy      0.944
  Bal. accuracy 0.816
  MCC           0.895
  Per class     negative          P 0.950  R 0.962  F1 0.956  (n=  705, predicted   714)
                neutral           P 0.655  R 0.521  F1 0.580  (n=   73, predicted    58)
                positive          P 0.959  R 0.966  F1 0.963  (n=  805, predicted   811)

[p4-sent-xlmr-base-seg_pyvi-base-s31337-val]  xlmr-base / sentiment / base / seed 31337


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2401.96it/s]


  epoch 1/4  loss 0.4834  dev macro-F1 0.7196  wtd 0.9119  acc 0.9242  neutral F1 0.273  (78s)
  epoch 2/4  loss 0.2386  dev macro-F1 0.8332  wtd 0.9388  acc 0.9419  neutral F1 0.588  (157s)
  epoch 3/4  loss 0.1757  dev macro-F1 0.8373  wtd 0.9405  acc 0.9438  neutral F1 0.598  (236s)
  epoch 4/4  loss 0.1424  dev macro-F1 0.8278  wtd 0.9384  acc 0.9425  neutral F1 0.571  (315s)
  --> validation
  Macro-F1      0.837   [95% CI 0.800-0.873]
  Weighted F1   0.940
  Accuracy      0.944
  Bal. accuracy 0.804
  MCC           0.895
  Per class     negative          P 0.943  R 0.965  F1 0.954  (n=  705, predicted   721)
                neutral           P 0.795  R 0.479  F1 0.598  (n=   73, predicted    44)
                positive          P 0.952  R 0.968  F1 0.960  (n=  805, predicted   818)

=== SENTIMENT / xlmr-base / base / 5 seeds ===
  [validation]  n_seeds=5  best_epochs=[3, 4, 4, 4, 3]
    macro_f1       0.8403 ± 0.0063   [0.8329, 0.8499]
    weighted_f1    0.9421 ± 0.0019   [0.940

CompletedProcess(args=['/usr/bin/python3', '-m', 'vifeedback.cli', 'train', 'run', '--task', 'sentiment', '--model', 'xlmr-base', '--recipe', 'base', '--preprocessing', 'seg_pyvi', '--seeds', 'all', '--epochs', '4', '--batch-size', '32', '--max-length', '96', '--phase', '4'], returncode=0)

In [15]:
# 4c — CafeBERT (XLM-R-large continued on 18GB Vietnamese, 560M params, ~11 GB).
# Needs registering in constants.MODEL_IDS first; skip unless you have added it.
# !python -m vifeedback.cli train run --task sentiment --model cafebert \
#     --preprocessing seg_pyvi --seeds all --batch-size 8 --grad-accum 4 --lr 8e-6 --phase 4


---

## 4d. Export the serving artifact (ONNX)

Runs here because Windows Application Control blocks the `onnx` native extension on the
reference machine (ADR-017). `onnxruntime` itself works there, so the split is
**export on Linux, benchmark on the reference CPU** — an `.onnx` graph is deterministic and
hardware-independent, a latency number is not.

Train the champion with `--save-checkpoint` first, or point `CKPT` at a checkpoint that came
back in an earlier results archive.


In [17]:
import pathlib
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check',
    'onnx>=1.17', 'onnxscript>=0.1', 'onnxruntime>=1.19'
], check=True)

# Train the model whose artifact ships, keeping the weights.
subprocess.run([
    sys.executable, '-m', 'vifeedback.cli', 'train', 'run',
    '--task', 'sentiment', '--model', 'phobert-base',
    '--preprocessing', 'seg_pyvi', '--seeds', '42',
    '--epochs', '4', '--save-checkpoint', '--phase', '6',
], check=True)

CKPT = 'models/p6-sent-phobert-base-seg_pyvi-base-s42-ckp'


# ============================================================
# Detect whether this version of vifeedback provides export CLI
# ============================================================

def command_exists(args):
    result = subprocess.run(
        [sys.executable, '-m', 'vifeedback.cli', *args, '--help'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    return result.returncode == 0


export_cmd = None

# Newer layout: vifeedback.cli serve export
if command_exists(['serve']):
    if command_exists(['serve', 'export']):
        export_cmd = ['serve', 'export']

# Alternative layout: vifeedback.cli export
if export_cmd is None and command_exists(['export']):
    export_cmd = ['export']


# ============================================================
# Export only when supported by the installed CLI
# ============================================================

if export_cmd is not None:
    print('Using export command:', ' '.join(export_cmd))

    # Export -> graph-optimize -> quantize -> VERIFY PARITY.
    for quant in ('dynamic', 'static'):
        print(f'--- {quant} ---')

        subprocess.run([
            sys.executable, '-m', 'vifeedback.cli',
            *export_cmd,
            '--checkpoint', CKPT,
            '--task', 'sentiment',
            '--quantize', quant,
            '--out', f'models/serve/sentiment_{quant}',
        ], check=True)

    subprocess.run([
        sys.executable, '-m', 'vifeedback.cli',
        *export_cmd,
        '--checkpoint', CKPT,
        '--task', 'sentiment',
        '--quantize', 'none',
        '--out', 'models/serve/sentiment_fp32',
    ], check=True)

else:
    print(
        '\n[WARNING] This vifeedback version does not provide '
        '`serve export` or `export` in vifeedback.cli.'
    )
    print('Training/checkpoint completed successfully.')
    print('Skipping ONNX export so this cell does not fail.')


# ============================================================
# Show exported model sizes, if any
# ============================================================

serve_dir = pathlib.Path('models/serve')

if serve_dir.exists():
    print('\nExported artifacts:')

    for p in sorted(serve_dir.iterdir()):
        if p.is_dir():
            size = sum(
                f.stat().st_size
                for f in p.rglob('*')
                if f.is_file()
            ) / (1024**2)

            print(f'{p}: {size:.1f} MiB')
else:
    print('\nNo models/serve directory was created.')

print('\nDone.')

  effective batch = 32 x 1 = 32

[p6-sent-phobert-base-seg_pyvi-base-s42-val]  phobert-base / sentiment / base / seed 42


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9738.56it/s]


  epoch 1/4  loss 0.3705  dev macro-F1 0.8384  wtd 0.9480  acc 0.9520  neutral F1 0.584  (41s)
  epoch 2/4  loss 0.1585  dev macro-F1 0.8505  wtd 0.9491  acc 0.9520  neutral F1 0.622  (82s)
  epoch 3/4  loss 0.1178  dev macro-F1 0.8498  wtd 0.9504  acc 0.9533  neutral F1 0.617  (126s)
  epoch 4/4  loss 0.0855  dev macro-F1 0.8644  wtd 0.9521  acc 0.9539  neutral F1 0.661  (169s)
  --> validation
  Macro-F1      0.864   [95% CI 0.829-0.896]
  Weighted F1   0.952
  Accuracy      0.954
  Bal. accuracy 0.840
  MCC           0.914
  Per class     negative          P 0.951  R 0.973  F1 0.962  (n=  705, predicted   721)
                neutral           P 0.778  R 0.575  F1 0.661  (n=   73, predicted    54)
                positive          P 0.968  R 0.971  F1 0.970  (n=  805, predicted   808)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]


  checkpoint: /kaggle/working/repo/models/p6-sent-phobert-base-seg_pyvi-base-s42-ckp

=== SENTIMENT / phobert-base / base / 1 seeds ===
  [validation]  n_seeds=1  best_epochs=[4]
    macro_f1       0.8644 ± 0.0000   [0.8644, 0.8644]
    weighted_f1    0.9521 ± 0.0000   [0.9521, 0.9521]
    accuracy       0.9539 ± 0.0000   [0.9539, 0.9539]
    per-class F1   negative=0.962±0.000  neutral=0.661±0.000  positive=0.970±0.000

[WARNING] This vifeedback version does not provide `serve export` or `export` in vifeedback.cli.
Training/checkpoint completed successfully.
Skipping ONNX export so this cell does not fail.

No models/serve directory was created.

Done.


---

## 5. Verify before you leave

Confirms runs actually landed in the registry. A session that finished without writing rows has
produced nothing, and it is much cheaper to notice that here than after the session expires.


In [21]:
import importlib
import pathlib
import pandas as pd

registry_path = WORK / 'results' / 'registry.csv'

def load_registry_compat() -> pd.DataFrame:
    """
    Prefer the repository's loader when it exists.
    Fall back to results/registry.csv for older repository versions that do not
    provide vifeedback.evaluation.report.
    """
    try:
        report = importlib.import_module('vifeedback.evaluation.report')
        loader = getattr(report, 'load_registry', None)
        if callable(loader):
            print('registry loader: vifeedback.evaluation.report.load_registry')
            return loader()
    except (ModuleNotFoundError, ImportError, AttributeError) as e:
        print(f'repository registry loader unavailable ({type(e).__name__}: {e})')

    if registry_path.is_file():
        print('registry loader: direct CSV ->', registry_path)
        try:
            return pd.read_csv(registry_path)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()

    print('WARNING: registry.csv does not exist yet:', registry_path)
    return pd.DataFrame()

reg = load_registry_compat()
print(f'total registry rows: {len(reg)}')

model_pattern = r'phobert-large|xlmr-base|xlmr|cafebert'
mask = pd.Series(False, index=reg.index, dtype=bool)

if 'model' in reg.columns:
    mask |= reg['model'].astype(str).str.contains(model_pattern, case=False, regex=True, na=False)
if 'run_id' in reg.columns:
    mask |= reg['run_id'].astype(str).str.contains(model_pattern, case=False, regex=True, na=False)

new = reg.loc[mask].copy()
print(f'rows matching Kaggle-target models: {len(new)}')

if reg.empty:
    print(
        'No registry rows are available. If training has not run yet, this is expected. '
        'If training finished, inspect the training output and the repository results path.'
    )
elif len(new) == 0:
    print(
        'WARNING: no registry rows matched the expected model names. '
        'Showing recent rows so the actual schema/names are visible.'
    )
    display(reg.tail(min(20, len(reg))))
else:
    preferred = ['run_id', 'task', 'model', 'split', 'macro_f1', 'weighted_f1', 'accuracy']
    cols = [c for c in preferred if c in new.columns]
    display(new[cols].tail(50) if cols else new.tail(50))


registry loader: vifeedback.evaluation.report.load_registry
total registry rows: 79
rows matching Kaggle-target models: 10


,run_id,task,model,split,macro_f1,weighted_f1,accuracy
67,p4-sent-phobert-large-seg_pyvi-base-s42-val,sentiment,phobert-large,validation,0.8604,0.9513,0.9539
68,p4-sent-phobert-large-seg_pyvi-base-s1337-val,sentiment,phobert-large,validation,0.8520,0.9507,0.9533
69,p4-sent-phobert-large-seg_pyvi-base-s2024-val,sentiment,phobert-large,validation,0.8555,0.9508,0.9533
70,p4-sent-phobert-large-seg_pyvi-base-s7-val,sentiment,phobert-large,validation,0.8534,0.9506,0.9533
71,p4-sent-phobert-large-seg_pyvi-base-s31337-val,sentiment,phobert-large,validation,0.8589,0.9510,0.9533
72,p4-sent-xlmr-base-seg_pyvi-base-s42-val,sentiment,xlmr-base,validation,0.8393,0.9411,0.9444
73,p4-sent-xlmr-base-seg_pyvi-base-s1337-val,sentiment,xlmr-base,validation,0.8499,0.9453,0.9476
74,p4-sent-xlmr-base-seg_pyvi-base-s2024-val,sentiment,xlmr-base,validation,0.8420,0.9417,0.9444
75,p4-sent-xlmr-base-seg_pyvi-base-s7-val,sentiment,xlmr-base,validation,0.8329,0.9420,0.9438
76,p4-sent-xlmr-base-seg_pyvi-base-s31337-val,sentiment,xlmr-base,validation,0.8373,0.9405,0.9438


In [22]:
# Mean +/- std per configuration when the expected columns exist.
required = {'task', 'model', 'split', 'macro_f1'}

if len(new) and required.issubset(new.columns):
    val = new[new['split'].astype(str).str.lower().isin(['validation', 'val', 'dev'])]
    if len(val):
        display(
            val.groupby(['task', 'model'])['macro_f1']
            .agg(['count', 'mean', 'std', 'min', 'max'])
            .round(4)
        )
    else:
        print('No validation/dev rows found among the matched runs.')
else:
    missing = sorted(required.difference(new.columns))
    print('Summary skipped; missing required columns:', missing if missing else '(no matched rows)')


count    mean     std     min     max
task      model                                               
sentiment phobert-large      5  0.8560  0.0036  0.8520  0.8604
          xlmr-base          5  0.8403  0.0063  0.8329  0.8499

---

## 6. Bring the results home

`/kaggle/working` is saved with the notebook version, so **Save Version → Output** already
persists everything. The zip below is for downloading it as one file.


In [23]:
import pathlib
import shutil

results_dir = WORK / 'results'
if not results_dir.exists():
    raise FileNotFoundError(
        f'{results_dir} does not exist. Run the training/evaluation cells first. '
        'If the repository writes results elsewhere, update results_dir in this cell.'
    )

archive_base = pathlib.Path('/kaggle/working/kaggle_results')
archive_path = pathlib.Path(
    shutil.make_archive(str(archive_base), 'zip', root_dir=str(results_dir))
)
size_mb = archive_path.stat().st_size / 1e6
print(f'{archive_path}  ({size_mb:.1f} MB)')
print('Download it from the Output panel on the right after the cell finishes.')


/kaggle/working/kaggle_results.zip  (0.8 MB)
Download it from the Output panel on the right after the cell finishes.


### Merging on the laptop

Merge **by `run_id`**, never by blind append — the archive's `registry.csv` also contains rows
that were already committed before you built the dataset.

```bash
unzip -o kaggle_results.zip -d /tmp/kag
cp -rn /tmp/kag/runs/* results/runs/
python - <<'EOF'
import pandas as pd
a = pd.read_csv('results/registry.csv')
b = pd.read_csv('/tmp/kag/registry.csv')
out = pd.concat([a, b]).drop_duplicates(subset='run_id', keep='first')
out.to_csv('results/registry.csv', index=False)
print(f'{len(out) - len(a)} new rows merged')
EOF
```

### Reporting the split honestly

Rows produced here carry a different `env.json` — P100, not RTX 3050. That is irrelevant for
**accuracy**, which is hardware-independent, and disqualifying for **latency**, which is not.
Mark Kaggle-trained rows in the final results table and name the GPU.
